In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
%sql
--drop table training.delta_demo.people_delta_demo;

In [0]:
%sql
create catalog if not exists training;
create schema if not exists training.delta_demo;
    
 

In [0]:
# DBTITLE 1,Create Delta Table
from pyspark.sql import Row

# Create initial DataFrame with sample data
data = [Row(id=1, name='Alice', age=30), Row(id=2, name='Bob', age=25), Row(id=3, name='Charlie', age=35)]
df = spark.createDataFrame(data)

# Overwrite and create a managed Delta table with the DataFrame in the training.delta_demo schema
df.write.format('delta').mode('overwrite').saveAsTable('training.delta_demo.people_delta_demo')

# Read the Delta table to verify creation and display its contents
delta_df = spark.read.table("training.delta_demo.people_delta_demo")
display(delta_df)

In [0]:
df.write.mode("append").format("delta").save("/Volumes/training/delta_demo/training/mytable")

In [0]:
%sql
describe history '/Volumes/training/delta_demo/training/mytable/'

In [0]:
df2 = spark.read.load("/Volumes/training/delta_demo/training/mytable")
display(df2)

In [0]:
data = [Row(id=1, name='Tom'), Row(id=8, name='Herry'), Row(id=9, name='Jerry'),Row(id=7, name='Pavan')]
df = spark.createDataFrame(data).withColumn("country", lit("USA"))
display(df)

In [0]:
#it will thrown an because of Schema Evaluation
#df.write.mode("overwrite").format("delta").save("/Volumes/training/delta_demo/training/mytable")
#by adding MergeSchema-->true it will solve the issue it is know as Schema Enforcement
df.write.mode("append").option("mergeSchema", "true").format("delta").save("/Volumes/training/delta_demo/training/mytable")

In [0]:
display(df)

In [0]:
df2= spark.read.load("/Volumes/training/delta_demo/training/mytable")
df2.display()

In [0]:
df2.write.mode("append").option("mergeSchema", "true").format("delta").saveAsTable("training.delta_demo.people_delta_demo")

In [0]:
%sql
describe history training.delta_demo.people_delta_demo;

In [0]:
%sql
select * from training.delta_demo.people_delta_demo version as of 2;

In [0]:
%sql
restore training.delta_demo.people_delta_demo version as of 1;

In [0]:
%sql
describe history training.delta_demo.people_delta_demo;

In [0]:
data = [Row(id=1, name='Tom', age=28), Row(id=7, name='Herry', age=27), Row(id=8, name='Jerry', age=37)]
df = spark.createDataFrame(data)
display(df)

In [0]:
df.createOrReplaceTempView("updates")

In [0]:
merge_sql = """
MERGE INTO training.delta_demo.people_delta_demo AS target
USING updates AS source
ON target.id = source.id
WHEN MATCHED THEN
  UPDATE SET target.name = source.name, target.age = source.age
WHEN NOT MATCHED THEN
  INSERT (id, name, age) VALUES (source.id, source.name, source.age)
"""

spark.sql(merge_sql)

In [0]:
%sql
select * from training.delta_demo.people_delta_demo;
    
 

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Delta Table Optimization Notebook
# MAGIC
# MAGIC This notebook focuses on key Delta Lake maintenance operations:
# MAGIC
# MAGIC - **Optimizing Tables**: Use the OPTIMIZE command to compact small files and improve query performance.
# MAGIC - **ZORDER Optimization**: Apply ZORDER BY to colocate related data, enabling efficient data skipping and faster queries.
# MAGIC - **Running VACUUM**: Remove obsolete files and reclaim storage space by executing the VACUUM command.
# MAGIC
# MAGIC These features help maintain efficient, performant Delta tables in your data lake.

In [0]:
%sql
drop table if exists training.delta_demo.people_delta_demo

In [0]:
from pyspark.sql import Row

# Create initial DataFrame with sample data
data = [Row(id=1, name='Alice', age=30), Row(id=2, name='Bob', age=25), Row(id=3, name='Charlie', age=35)]
df = spark.createDataFrame(data)

# Overwrite and create a managed Delta table with the DataFrame in the training.delta_demo schema
df.write.format('delta').mode('overwrite').saveAsTable('training.delta_demo.people_delta_demo')

# Read the Delta table to verify creation and display its contents
delta_df = spark.read.table('training.delta_demo.people_delta_demo')
display(delta_df)

In [0]:
df.write.format("delta").mode("append").save("/Volumes/training/delta_demo/training/mytable")

In [0]:
df2=spark.read.format('delta').load(
    "/Volumes/training/delta_demo/training/mytable"
)
display(df2)

In [0]:
%sql
optimize "/Volumes/training/delta_demo/training/mytable"

In [0]:
%sql
describe history "/Volumes/training/delta_demo/training/mytable"

In [0]:
%sql
optimize "/Volumes/training/delta_demo/training/mytable" zorder by id;

In [0]:
df2=spark.read.format('delta').load(
    "/Volumes/training/delta_demo/training/mytable"
)
display(df2)

In [0]:
%sql
DESCRIBE DETAIL training.delta_demo.people_delta_demo;

In [0]:
%sql
vacuum training.delta_demo.people_delta_demo dry run 

--- IT WILL NOT DELETE ANYTHING
    
--vacuum training.delta_demo.people_delta_demo retain 0 hours
    
----describe history training.delta_demo.people_delta_demo
    
--drop table if exists training.delta_demo.people_delta_demo
    
--describe history "/Volumes/training/delta_demo/training/mytable"

In [0]:
%sql
SELECT * FROM training.delta_demo.people_delta_demo

In [0]:
%sql
describe history  training.delta_demo.people_delta_demo

In [0]:
%sql
vacuum training.delta_demo.people_delta_demo retain 0 hours
---TO RUN THIS WE NEED TO MAKE SOME CONFIGURATION SETUP FOR VACCUN

In [0]:
%sql
--SET spark.databricks.delta.retentionDurationCheck.enabled = false;
-- IN FREE VERSION THIS OPTION IS NOT AVAILABLE

In [0]:
 %sql
----SET 'delta.logRetentionDuration' = 'interval 30 days',
--Yes—d--elta.logRetentionDuration controls cleanup of old Delta transaction-log files, which are mainly the versioned JSON commit files in the table’s _delta_log folder.
--SET 'delta.deletedFileRetentionDuration' = 'interval 7 days'
 --then it does not immediately delete Parquet files. It only sets the eligibility window: an obsolete Delta data file must remain for at least 7 days before VACUUM is allowed to physically remove it.
    
 